### Django Real-World Usage
1. Replace Repetitive Model Field Validation

In [ ]:
# descriptors.py (shared across Django app)
class PositiveDecimal:
    def __set_name__(self, owner, name):
        self.name         = name
        self.storage_name = f"_desc_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None: return self
        return getattr(obj, self.storage_name, None)

    def __set__(self, obj, value):
        from decimal import Decimal, InvalidOperation
        try:
            value = Decimal(str(value))
        except InvalidOperation:
            raise ValueError(f"{self.name!r} must be a valid number")
        if value <= 0:
            raise ValueError(f"{self.name!r} must be positive, got {value}")
        setattr(obj, self.storage_name, value)


# models.py
from django.db import models
from .descriptors import PositiveDecimal, NonEmptyString

class Product(models.Model):
    # Django fields handle DB — descriptors handle Python-level validation
    _name     = models.CharField(max_length=255, db_column="name")
    _price    = models.DecimalField(max_digits=10, decimal_places=2, db_column="price")
    _discount = models.DecimalField(max_digits=5,  decimal_places=2, db_column="discount")

    name     = NonEmptyString()
    price    = PositiveDecimal()
    discount = PositiveDecimal()

    class Meta:
        db_table = "products"

2. Audit / Dirty Tracking Descriptor

In [ ]:
class Audited:
    """
    Tracks when a field changes — useful for audit logs.
    Stores original value alongside current value.
    """
    def __set_name__(self, owner, name):
        self.name          = name
        self.current_key   = f"_current_{name}"
        self.original_key  = f"_original_{name}"
        self.is_dirty_key  = f"_dirty_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None: return self
        return getattr(obj, self.current_key, None)

    def __set__(self, obj, value):
        # On first set (object creation) — record original
        if not hasattr(obj, self.original_key):
            setattr(obj, self.original_key, value)
            setattr(obj, self.is_dirty_key,  False)
        else:
            original = getattr(obj, self.original_key)
            setattr(obj, self.is_dirty_key, value != original)
        setattr(obj, self.current_key, value)

    def is_dirty(self, obj) -> bool:
        return getattr(obj, self.is_dirty_key, False)


# models.py
class UserProfile(models.Model):
    email = Audited()
    role  = Audited()

    def get_dirty_fields(self) -> dict:
        dirty = {}
        for name, descriptor in type(self).__dict__.items():
            if isinstance(descriptor, Audited) and descriptor.is_dirty(self):
                dirty[name] = {
                    "from": getattr(self, descriptor.original_key),
                    "to":   getattr(self, descriptor.current_key),
                }
        return dirty


# Usage in a Django view / signal:
user = UserProfile(email="alice@old.com", role="viewer")
user.email = "alice@new.com"
user.role  = "viewer"   # unchanged

print(user.get_dirty_fields())
# {"email": {"from": "alice@old.com", "to": "alice@new.com"}}
# ↑ Only email is dirty — role unchanged → clean audit log for free